# QUESTÕES DE IMPLEMENTAÇÃO DE CÓDIGO AC01

In [3]:
#### 5 ######
import random

# =====================================================================
# 1. ESTRUTURA DA LISTA ENCADEADA SIMPLES (PARA AS LIGAÇÕES / ARESTAS)
# =====================================================================

class NodoAresta:
    """Nó da Lista Encadeada Simples de Arestas"""
    def __init__(self, id_aresta, destino, custo=1.0, caracteristica=""):
        self.id_aresta = id_aresta
        self.destino = destino
        self.custo = custo
        self.caracteristica = caracteristica
        self.prox = None  # Ponteiro para o próximo nó da lista encadeada simples

class ListaEncadeadaArestas:
    """Lista Encadeada Simples de adjacências de um vértice"""
    def __init__(self):
        self.head = None

    def inserir(self, id_aresta, destino, custo, caracteristica):
        novo_no = NodoAresta(id_aresta, destino, custo, caracteristica)
        novo_no.prox = self.head
        self.head = novo_no

    def buscar(self, id_aresta):
        atual = self.head
        while atual:
            if atual.id_aresta == id_aresta:
                return atual
            atual = atual.prox
        return None

    def alterar(self, id_aresta, novo_custo=None, nova_caracteristica=None):
        no = self.buscar(id_aresta)
        if no:
            if novo_custo is not None:
                no.custo = novo_custo
            if nova_caracteristica is not None:
                no.caracteristica = nova_caracteristica
            return True
        return False

    def remover(self, id_aresta):
        atual = self.head
        anterior = None
        while atual:
            if atual.id_aresta == id_aresta:
                if anterior is None:
                    self.head = atual.prox
                else:
                    anterior.prox = atual.prox
                return True
            anterior = atual
            atual = atual.prox
        return False

    def remover_por_destino(self, destino_id):
        """Remove todas as ligações que apontam para um vértice excluído"""
        atual = self.head
        anterior = None
        while atual:
            if atual.destino == destino_id:
                if anterior is None:
                    self.head = atual.prox
                    atual = self.head
                else:
                    anterior.prox = atual.prox
                    atual = anterior.prox
            else:
                anterior = atual
                atual = atual.prox

    def para_lista(self):
        arestas = []
        atual = self.head
        while atual:
            arestas.append(atual)
            atual = atual.prox
        return arestas


# =====================================================================
# 2. ESTRUTURA DO VÉRTICE E DO MULTIGRAFO
# =====================================================================

class Vertice:
    """Elemento do Vetor de Vértices"""
    def __init__(self, id_vertice, rotulo="", custo=0.0):
        self.id_vertice = id_vertice
        self.rotulo = rotulo
        self.custo = custo
        self.lista_adj = ListaEncadeadaArestas()  # Aponta para sua Lista Encadeada

class Multigrafo:
    """Classe Multigrafo baseada em Vetor de Listas Encadeadas Simples"""
    def __init__(self, nome="Multigrafo"):
        self.nome = nome
        self.vertices = []  # Vetor de objetos Vertice
        self.proximo_id_aresta = 1

    def destruir(self):
        """Destrói a estrutura e esvazia o grafo"""
        self.vertices.clear()
        self.proximo_id_aresta = 1

    # --- OPERAÇÕES DE VÉRTICES ---

    def incluir_vertice(self, id_vertice, rotulo="", custo=0.0):
        if self.buscar_vertice(id_vertice) is not None:
            return False
        novo_v = Vertice(id_vertice, rotulo, custo)
        self.vertices.append(novo_v)
        return True

    def buscar_vertice(self, id_vertice):
        for v in self.vertices:
            if v.id_vertice == id_vertice:
                return v
        return None

    def alterar_vertice(self, id_vertice, novo_rotulo=None, novo_custo=None):
        v = self.buscar_vertice(id_vertice)
        if v:
            if novo_rotulo is not None:
                v.rotulo = novo_rotulo
            if novo_custo is not None:
                v.custo = novo_custo
            return True
        return False

    def remover_vertice(self, id_vertice):
        v = self.buscar_vertice(id_vertice)
        if not v:
            return False
        
        # Remove o vértice do vetor
        self.vertices.remove(v)

        # Remove todas as arestas de outros vértices que apontavam para o vértice excluído
        for outro_v in self.vertices:
            outro_v.lista_adj.remover_por_destino(id_vertice)
        return True

    # --- OPERAÇÕES DE LIGAÇÕES / ARESTAS (SUPORTA MULTIPLAS LIGAÇÕES) ---

    def incluir_aresta(self, origem_id, destino_id, custo=1.0, caracteristica="", id_aresta=None):
        v_origem = self.buscar_vertice(origem_id)
        v_destino = self.buscar_vertice(destino_id)

        if not v_origem or not v_destino:
            return None

        if id_aresta is None:
            id_aresta = self.proximo_id_aresta
            self.proximo_id_aresta += 1

        # Inserção na lista encadeada simples da origem (admite múltiplas arestas entre u e v)
        v_origem.lista_adj.inserir(id_aresta, destino_id, custo, caracteristica)
        return id_aresta

    def buscar_aresta(self, id_aresta):
        for v in self.vertices:
            aresta = v.lista_adj.buscar(id_aresta)
            if aresta:
                return aresta, v.id_vertice
        return None, None

    def alterar_aresta(self, id_aresta, novo_custo=None, nova_caracteristica=None):
        aresta, origem_id = self.buscar_aresta(id_aresta)
        if aresta:
            v_origem = self.buscar_vertice(origem_id)
            return v_origem.lista_adj.alterar(id_aresta, novo_custo, nova_caracteristica)
        return False

    def remover_aresta(self, id_aresta):
        aresta, origem_id = self.buscar_aresta(id_aresta)
        if aresta:
            v_origem = self.buscar_vertice(origem_id)
            return v_origem.lista_adj.remover(id_aresta)
        return False

    # --- MOSTRAR E PREENCHIMENTO ---

    def mostrar(self):
        print(f"\n=================== {self.nome} ===================")
        if not self.vertices:
            print("Grafo vazio.")
            print("=====================================================")
            return

        print(f"Total de Vértices: {len(self.vertices)}")
        for v in self.vertices:
            print(f"\n-> [Vértice ID: {v.id_vertice}] | Rótulo: '{v.rotulo}' | Custo: {v.custo}")
            arestas = v.lista_adj.para_lista()
            if not arestas:
                print("   └── (Sem ligações de saída)")
            else:
                for a in arestas:
                    print(f"   └── (Aresta ID: {a.id_aresta}) ---> Destino: {a.destino} | Custo: {a.custo} | Caract: '{a.caracteristica}'")
        print("=====================================================")

    def preenchimento_automatico(self, num_vertices=4, num_arestas=6):
        """Gera um grafo automaticamente com atributos aleatórios"""
        self.destruir()
        
        tipos_v = ["Servidor", "Roteador", "Switch", "DataCenter"]
        for i in range(1, num_vertices + 1):
            self.incluir_vertice(i, f"{random.choice(tipos_v)}_{i}", round(random.uniform(10, 50), 2))

        tipos_a = ["Fibra", "Rádio", "Cabo", "Satélite"]
        for _ in range(num_arestas):
            u = random.randint(1, num_vertices)
            v = random.randint(1, num_vertices)
            self.incluir_aresta(u, v, round(random.uniform(1.0, 20.0), 2), random.choice(tipos_a))

    def obter_estatisticas(self):
        total_v = len(self.vertices)
        total_a = 0
        custo_a = 0.0
        for v in self.vertices:
            arestas = v.lista_adj.para_lista()
            total_a += len(arestas)
            custo_a += sum(a.custo for a in arestas)
        
        return {
            "Vertices": total_v,
            "Arestas": total_a,
            "Custo Total Arestas": round(custo_a, 2),
            "Grau Médio Saída": round(total_a / total_v, 2) if total_v > 0 else 0
        }


# =====================================================================
# 3. FUNÇÃO DE COMPARAÇÃO ENTRE GRAFOS
# =====================================================================

def comparar_grafos(g1: Multigrafo, g2: Multigrafo):
    e1 = g1.obter_estatisticas()
    e2 = g2.obter_estatisticas()
    
    print("\n" + "="*55)
    print(f"          COMPARAÇÃO: {g1.nome} vs {g2.nome}")
    print("="*55)
    print(f"{'Métrica':<25} | {g1.nome:<12} | {g2.nome:<12}")
    print("-" * 55)
    for chave in e1:
        print(f"{chave:<25} | {str(e1[chave]):<12} | {str(e2[chave]):<12}")
    print("="*55)


#======== IMPLEMENTAÇÃO ==========#
# 1. Criando Grafo 1 (Manual)
g1 = Multigrafo("Grafo_Rede_Manual")

# Inserção manual de Vértices
g1.incluir_vertice(1, "Roteador_Central", 150.0)
g1.incluir_vertice(2, "Switch_A", 80.0)
g1.incluir_vertice(3, "Servidor_Web", 200.0)

# Inserção manual de Múltiplas Ligações (Multigrafo) entre os mesmos nós
a1 = g1.incluir_aresta(1, 2, custo=10.5, caracteristica="Link Principal Fibra")
a2 = g1.incluir_aresta(1, 2, custo=25.0, caracteristica="Link Redundante Rádio")
a3 = g1.incluir_aresta(2, 3, custo=5.0, caracteristica="Cabo UTP Cat6")

# Alteração de dados
g1.alterar_vertice(2, novo_rotulo="Switch_A_Atualizado", novo_custo=95.0)
g1.alterar_aresta(a1, novo_custo=8.0, nova_caracteristica="Fibra Optica 10Gbps")

# Exibição do Grafo 1
g1.mostrar()

# 2. Criando Grafo 2 (Automático)
g2 = Multigrafo("Grafo_Rede_Auto")
g2.preenchimento_automatico(num_vertices=5, num_arestas=10)

# Exibição do Grafo 2
g2.mostrar()

# 3. Comparação Direta
comparar_grafos(g1, g2)


=================== Grafo_Rede_Manual ===================
Total de Vértices: 3

-> [Vértice ID: 1] | Rótulo: 'Roteador_Central' | Custo: 150.0
   └── (Aresta ID: 2) ---> Destino: 2 | Custo: 25.0 | Caract: 'Link Redundante Rádio'
   └── (Aresta ID: 1) ---> Destino: 2 | Custo: 8.0 | Caract: 'Fibra Optica 10Gbps'

-> [Vértice ID: 2] | Rótulo: 'Switch_A_Atualizado' | Custo: 95.0
   └── (Aresta ID: 3) ---> Destino: 3 | Custo: 5.0 | Caract: 'Cabo UTP Cat6'

-> [Vértice ID: 3] | Rótulo: 'Servidor_Web' | Custo: 200.0
   └── (Sem ligações de saída)

=================== Grafo_Rede_Auto ===================
Total de Vértices: 5

-> [Vértice ID: 1] | Rótulo: 'Switch_1' | Custo: 41.4
   └── (Aresta ID: 9) ---> Destino: 1 | Custo: 8.42 | Caract: 'Cabo'

-> [Vértice ID: 2] | Rótulo: 'Roteador_2' | Custo: 27.72
   └── (Aresta ID: 8) ---> Destino: 2 | Custo: 4.11 | Caract: 'Cabo'
   └── (Aresta ID: 6) ---> Destino: 1 | Custo: 7.06 | Caract: 'Fibra'
   └── (Aresta ID: 5) ---> Destino: 5 | Custo: 14.19 |

In [4]:
#==========6===========#


from collections import defaultdict

class GrafoDAG:
    def __init__(self, num_vertices):
        self.V = num_vertices
        # Lista de Adjacência usando dicionário de listas
        self.adj = defaultdict(list)

    def adicionar_aresta(self, u, v):
        self.adj[u].append(v)

    def _dfs_topologica(self, v, visitado, pilha):
        visitado[v] = True
        
        # Visita recursivamente todos os vértices dependentes
        for vizinho in self.adj[v]:
            if not visitado[vizinho]:
                self._dfs_topologica(vizinho, visitado, pilha)
                
        # Empilha o vértice apenas após processar todos os seus descendentes (pós-ordem)
        pilha.append(v)

    def ordenacao_topologica(self):
        visitado = [False] * self.V
        pilha = []

        # Executa a busca em profundidade para todos os componentes desconexos
        for i in range(self.V):
            if not visitado[i]:
                self._dfs_topologica(i, visitado, pilha)

        # O resultado topológico é a ordem de desempilhamento (pós-ordem invertida)
        return pilha[::-1]


g = GrafoDAG(15)
arestas = [
    (0, 1), (0, 2), (1, 3), (1, 4), (2, 5), (2, 6),
    (3, 7), (4, 7), (4, 8), (5, 8), (5, 9), (6, 9),
    (7, 10), (8, 10), (8, 11), (9, 11), (10, 12), (11, 13),
    (12, 14), (13, 14)
]
for u, v in arestas:
    g.adicionar_aresta(u, v)
rotulacao = g.ordenacao_topologica()
print("Resultado da Rotulação Topológica:", rotulacao)

Resultado da Rotulação Topológica: [0, 2, 6, 5, 9, 1, 4, 8, 11, 13, 3, 7, 10, 12, 14]
